In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")


Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze (
    kafka_key STRING,
    raw_value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    kafka_timestamp TIMESTAMP
) USING iceberg
""")

DataFrame[]

In [3]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [4]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [6]:
# SEMINAR TASK 
# Consume a few messages from the topic using kafka-console-consumer.sh to verify they are there

# docker exec kafka sh -c "/opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic taxi-trips --from-beginning  --max-messages 5"


In [ ]:
# starts the query
bronze = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_timestamp")
    )
)

query = (
    bronze.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/taxi_bronze")
    .toTable("lakehouse.taxi.bronze")
)

query.awaitTermination()

In [6]:
# RUN THIS TO STOP THE QUERY
query.stop()

In [7]:
spark.sql("SELECT count(*) FROM lakehouse.taxi.bronze").show()
spark.sql("SELECT * FROM lakehouse.taxi.bronze LIMIT 10").show()

+--------+
|count(1)|
+--------+
|     454|
+--------+

+---------+--------------------+----------+---------+------+--------------------+
|kafka_key|           raw_value|     topic|partition|offset|     kafka_timestamp|
+---------+--------------------+----------+---------+------+--------------------+
|        1|{"VendorID": 1, "...|taxi-trips|        0|     0|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     1|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     2|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     3|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     4|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     5|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     6|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     7|2026-04-02 12:08:...|
|        1|{"VendorID": 1, "...|taxi-trips

[]